# 🗑️ Waste Text Classification — Simple RNN
**Input:** `processed/train.csv` · `processed/val.csv` · `processed/test.csv`  
**Classes:** glass | metal | paper | plastic (+ cardboard | trash إذا موجودين في البيانات)

In [ ]:
import subprocess, sys
def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['tensorflow', 'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'seaborn']:
    install(pkg)

print('✅ packages ready')

In [ ]:
import os, re, random, json, collections, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, SimpleRNN, Dense, Dropout, SpatialDropout1D
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print('TensorFlow :', tf.__version__)
print('GPU        :', bool(tf.config.list_physical_devices('GPU')))

## Step 1 — Load Data
> بنقرأ الـ CSVs الجاهزة اللي عملها الـ preprocessing notebook

In [ ]:
TRAIN_CSV = 'processed/train.csv'
VAL_CSV   = 'processed/val.csv'
TEST_CSV  = 'processed/test.csv'

# ── تأكد إن الملفات موجودة ─────────────────────────────────────────────────
for f in [TRAIN_CSV, VAL_CSV, TEST_CSV]:
    assert os.path.exists(f), f'❌ مش لاقي: {f}  — شغّل preprocessing notebook الأول'

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

print(f'Train : {len(train_df):,} rows')
print(f'Val   : {len(val_df):,} rows')
print(f'Test  : {len(test_df):,} rows')
print()
print('Columns:', list(train_df.columns))
print()
print('Classes:', sorted(train_df['label'].unique()))

In [ ]:
# ── اختار عمود النص ──────────────────────────────────────────────────────────
# الـ preprocessing بيعمل 'text_clean' — لو مش موجود بنرجع لـ 'text_description'
TEXT_COL = 'text_clean' if 'text_clean' in train_df.columns else 'text_description'
print(f'Using text column: {TEXT_COL}')

# تأكد مفيش null
train_df[TEXT_COL] = train_df[TEXT_COL].fillna('').astype(str)
val_df[TEXT_COL]   = val_df[TEXT_COL].fillna('').astype(str)
test_df[TEXT_COL]  = test_df[TEXT_COL].fillna('').astype(str)

print('Sample:')
print(train_df[TEXT_COL].iloc[0][:200])

## Step 2 — EDA

In [ ]:
train_df['text_len'] = train_df[TEXT_COL].apply(lambda x: len(x.split()))

colors = ['#4CAF50', '#2196F3', '#FF9800', '#9C27B0', '#F44336', '#00BCD4']
labels_list = sorted(train_df['label'].unique())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

counts = train_df['label'].value_counts()
bars = axes[0].bar(counts.index, counts.values,
                   color=colors[:len(counts)], edgecolor='white')
axes[0].bar_label(bars, fmt='%d', padding=4, fontweight='bold')
axes[0].set_title('Class Distribution (Train)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Class'); axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)
axes[0].spines[['top','right']].set_visible(False)

for i, lbl in enumerate(labels_list):
    axes[1].hist(train_df[train_df['label'] == lbl]['text_len'],
                 bins=20, alpha=0.6, label=lbl, color=colors[i % len(colors)])
axes[1].set_title('Text Length Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('# Tokens'); axes[1].set_ylabel('Count')
axes[1].legend(); axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()
print(train_df['text_len'].describe().round(2))

## Step 3 — Encode Labels

In [ ]:
le = LabelEncoder()
le.fit(train_df['label'])
NUM_CLASSES = len(le.classes_)

print('Classes   :', le.classes_)
print('Num classes:', NUM_CLASSES)
print('Mapping   :', dict(zip(le.classes_, le.transform(le.classes_))))

## Step 4 — Tokenization & Padding

In [ ]:
MAX_VOCAB = 5000
MAX_LEN   = 60
EMBED_DIM = 64

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
tokenizer.fit_on_texts(train_df[TEXT_COL])   # fit على train فقط

vocab_size = min(MAX_VOCAB, len(tokenizer.word_index) + 1)
print(f'Unique words : {len(tokenizer.word_index)} → using top {vocab_size}')

def encode(texts):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LEN, padding='post', truncating='post')

X_train = encode(train_df[TEXT_COL])
X_val   = encode(val_df[TEXT_COL])
X_test  = encode(test_df[TEXT_COL])

y_train = to_categorical(le.transform(train_df['label']), NUM_CLASSES)
y_val   = to_categorical(le.transform(val_df['label']),   NUM_CLASSES)
y_test  = to_categorical(le.transform(test_df['label']),  NUM_CLASSES)

print(f'X_train: {X_train.shape} | y_train: {y_train.shape}')
print(f'X_val  : {X_val.shape}   | y_val  : {y_val.shape}')
print(f'X_test : {X_test.shape}  | y_test : {y_test.shape}')

## Step 5 — Build Model

In [ ]:
def build_simplernn(vocab_size, embed_dim, max_len, num_classes):
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=embed_dim,
                  input_length=max_len, name='embedding'),
        SpatialDropout1D(0.2),
        SimpleRNN(128, return_sequences=True, dropout=0.2, name='rnn_1'),
        SimpleRNN(64,  dropout=0.2, name='rnn_2'),
        Dense(64, activation='relu', name='dense_1'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax', name='output'),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_simplernn(vocab_size, EMBED_DIM, MAX_LEN, NUM_CLASSES)
model.summary()

## Step 6 — Train

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=3,
                            restore_best_weights=True, verbose=1)
reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                patience=2, min_lr=1e-6, verbose=1)
EPOCHS     = 20
BATCH_SIZE = 32

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

## Step 7 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
epochs = range(1, len(history.history['loss']) + 1)

axes[0].plot(epochs, history.history['loss'],     'b-o', ms=4, label='Train')
axes[0].plot(epochs, history.history['val_loss'], 'r-o', ms=4, label='Val')
axes[0].set_title('Loss', fontweight='bold'); axes[0].legend()
axes[0].spines[['top','right']].set_visible(False)

axes[1].plot(epochs, history.history['accuracy'],     'b-o', ms=4, label='Train')
axes[1].plot(epochs, history.history['val_accuracy'], 'r-o', ms=4, label='Val')
axes[1].set_title('Accuracy', fontweight='bold'); axes[1].legend()
axes[1].spines[['top','right']].set_visible(False)

plt.suptitle('Simple RNN — Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## Step 8 — Evaluate

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Loss     : {test_loss:.4f}')
print(f'Test Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)')

y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
y_true = np.argmax(y_test, axis=1)

print()
print('Classification Report:')
print(classification_report(y_true, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_title('Confusion Matrix', fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

## Step 9 — Inference

In [ ]:
def predict_text(text):
    seq   = tokenizer.texts_to_sequences([text.lower()])
    pad   = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')
    proba = model.predict(pad, verbose=0)[0]
    pred  = le.classes_[np.argmax(proba)]
    print(f'Input      : {text[:80]}')
    print(f'Prediction : {pred.upper()}  ({proba.max()*100:.1f}%)')
    for cls, p in sorted(zip(le.classes_, proba), key=lambda x: -x[1]):
        bar = '█' * int(p * 30)
        print(f'  {cls:12s}  {bar:<30s}  {p*100:5.1f}%')
    print()

examples = [
    'broken glass bottle recycling silica cullet',
    'crumpled newspaper cardboard paper pulp waste',
    'crushed aluminum steel can metal scrap',
    'plastic polymer polyethylene bottle recycling',
]
for t in examples:
    print('-' * 55)
    predict_text(t)

## Step 10 — Save Model

In [ ]:
os.makedirs('model_output', exist_ok=True)

model.save('model_output/waste_simplernn.h5')

with open('model_output/tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

with open('model_output/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print('✅ Saved to model_output/')
print('   ├── waste_simplernn.h5')
print('   ├── tokenizer.pkl')
print('   └── label_encoder.pkl')